In [53]:
import requests
import os
from dotenv import load_dotenv
import xml.etree.ElementTree as ET
import pandas as pd
from datetime import datetime
from zoneinfo import ZoneInfo

In [54]:
load_dotenv()

True

In [55]:
api_key = os.getenv("ENTSOE_API_KEY")

In [56]:
payload={}
headers = {
  "SECURITY_TOKEN": api_key
}

In [57]:
def convert_local_midnight_to_utc(date_str):
    pure_date = datetime.strptime(date_str, "%Y-%m-%d").date()
    prague_tz = ZoneInfo("Europe/Prague")
    local_midnight = datetime.combine(pure_date, time.min, tzinfo=prague_tz)
    utc_dt = local_midnight.astimezone(ZoneInfo("UTC"))
    
    # ENTSO-E needs YYYYMMDDHHmm
    return utc_dt.strftime("%Y%m%d%H%M")

In [61]:
hourly_data = []

def parse_xml_response(data, xml_text, namespace):
    root = ET.fromstring(response.text)
    ns = {"ns": namespace}
    
    for period_block in root.findall(".//ns:Period", ns):
        day = period_block.find("ns:timeInterval/ns:start", ns).text
        
        for point in period_block.findall("ns:Point", ns):
            quantity = float(point.find("ns:quantity", ns).text)
            
            data.append(quantity)

In [62]:
# data for 2022
url = f"https://web-api.tp.entsoe.eu/api?documentType=A65&processType=A16&outBiddingZone_Domain=10YCZ-CEPS-----N&periodStart={convert_local_midnight_to_utc('2022-12-25')}&periodEnd={convert_local_midnight_to_utc('2023-01-01')}"
response = requests.request("GET", url, headers=headers, data=payload)

parse_xml_response(hourly_data, response.text, "urn:iec62325.351:tc57wg16:451-6:generationloaddocument:3:0")
print(len(hourly_data))

168


In [63]:
# data for 2023
url = f"https://web-api.tp.entsoe.eu/api?documentType=A65&processType=A16&outBiddingZone_Domain=10YCZ-CEPS-----N&periodStart={convert_local_midnight_to_utc('2023-01-01')}&periodEnd={convert_local_midnight_to_utc('2024-01-01')}"
response = requests.request("GET", url, headers=headers, data=payload)

parse_xml_response(hourly_data, response.text, "urn:iec62325.351:tc57wg16:451-6:generationloaddocument:3:0")
print(len(hourly_data))

8928


In [64]:
# data for 2024 until 2024-07-01
url = f"https://web-api.tp.entsoe.eu/api?documentType=A65&processType=A16&outBiddingZone_Domain=10YCZ-CEPS-----N&periodStart={convert_local_midnight_to_utc('2024-01-01')}&periodEnd={convert_local_midnight_to_utc('2024-06-30')}"
response = requests.request("GET", url, headers=headers, data=payload)

parse_xml_response(hourly_data, response.text, "urn:iec62325.351:tc57wg16:451-6:generationloaddocument:3:0")
print(len(hourly_data))

13271


In [31]:
# Entso-e struggles to return consistent data for this date (it is last day with hour period, 2024-07-01 is first day with 15min period)
last_hourly_day = [4795.78, 4672.16, 4622.35, 4570.81, 4614.12, 4478.1, 4674.85, 5171.84, 5771.08, 6237.85, 6540.96, 6747.07, 6659.57, 6574.33, 6501.41, 6390.03, 6150.8, 5992.14, 5960.04, 5920.94, 5875.47, 5819.34, 5761.21, 5417.37]

hourly_data = hourly_data + last_hourly_day
print(len(hourly_data))

13295


In [32]:
df = pd.DataFrame(index=range(len(hourly_data)))
df["last_week_load"] = hourly_data
print(len(df))

13295


In [33]:
df_quarterly = df.loc[df.index.repeat(4)].reset_index(drop=True)
print(len(df_quarterly))

53180


In [69]:
quarterhour_data = []

# data for the rest of 2024
url = f"https://web-api.tp.entsoe.eu/api?documentType=A65&processType=A16&outBiddingZone_Domain=10YCZ-CEPS-----N&periodStart={convert_local_midnight_to_utc('2024-07-01')}&periodEnd={convert_local_midnight_to_utc('2025-01-01')}"
response = requests.request("GET", url, headers=headers, data=payload)

parse_xml_response(quarterhour_data, response.text, "urn:iec62325.351:tc57wg16:451-6:generationloaddocument:3:0")
print(len(quarterhour_data))

17666


In [75]:
# data 2025
url = f"https://web-api.tp.entsoe.eu/api?documentType=A65&processType=A16&outBiddingZone_Domain=10YCZ-CEPS-----N&periodStart={convert_local_midnight_to_utc('2025-01-01')}&periodEnd={convert_local_midnight_to_utc('2025-12-25')}"
response = requests.request("GET", url, headers=headers, data=payload)

parse_xml_response(quarterhour_data, response.text, "urn:iec62325.351:tc57wg16:451-6:generationloaddocument:3:0")
print(len(quarterhour_data))

In [38]:
df_new = pd.DataFrame(index=range(len(quarterhour_data))) 
df_new ["last_week_load"] = quarterhour_data

result_df = pd.concat([df_quarterly, df_new], ignore_index=True)

In [39]:
result_df.to_csv("../data/czechia_load_last_week_15min.csv", sep=";", index=False)